In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import timm
import torch
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
from torch import nn
from torch.optim import Adam
from torchvision import models
import splitfolders
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm

import os
import shutil
#os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
label_dic = {
    0: "Healthy",
    1: "Common Rust",
    2: "Blight",
    3: "Gray Leaf Spot",
}

In [ ]:
train_df = pd.read_csv('./data/train.csv')

test_df = pd.read_csv('./data/random_submission.csv')

In [ ]:
# divide images of train set into four floders following by "label_dic"
for label in label_dic.keys():
    if os.path.isdir("./train/{}".format(label_dic[label])):
        pass
    else:
        os.makedirs('./train/{}'.format(label_dic[label]))
    for image_id in train_df[train_df["Label"]==label]["Image"]:
        shutil.copy('./data/train/{}.jpg'.format(image_id), './train/{}/{}.jpg'.format(label_dic[label], image_id))

In [ ]:
# divide images of test set into four floders following by "label_dic"
for label in label_dic.keys():
    if os.path.isdir("./test/{}".format(label_dic[label])):
        pass
    else:
        os.makedirs('./test/{}'.format(label_dic[label]))
    for image_id in test_df[test_df["Predicted"]==label]["Image"]:
        shutil.copy('./data/test/{}.jpg'.format(image_id), './test/{}/{}.jpg'.format(label_dic[label], image_id))

In [ ]:
# need validation set from dividing train set(8/2)
splitfolders.ratio("./train",
            output="splitted_data",
            seed=42,
            ratio=(.8, .2),
            group_prefix=None,
            move=False)

In [ ]:
# In addition to resizing to a fixed size, converting to PyTorch-compatible data, and normalization, additional transformation operations are required to achieve data augmentation.
train_transform = transforms.Compose([
    transforms.Resize(size=(224, 224)),
# The image is randomly flipped horizontally with a 50% probability. This is a data augmentation technique that helps the model learn invariance to image orientation.
    transforms.RandomHorizontalFlip(),
# The image is randomly flipped vertically with a 50% probability. This is another data augmentation technique.
    transforms.RandomVerticalFlip(),
# Apply random Gaussian blur to the image. This added noise helps the model become more robust in predicting slight variations in the image.
    transforms.GaussianBlur(kernel_size=(3, 7), sigma=(0.1, 2)),
# narray convert to PyTorch tensor
    transforms.ToTensor(),
# Normalize the tensor image using the mean and standard deviation.
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

#train_transform above has data augmentation, but Val and test transform below are ensured, not random transformations, ensure that the prediction results of the model are stable and reliable.If random enhancement is used, it may lead to inconsistent test results and affect the evaluation of model performance.
test_val_transform = transforms.Compose([
    transforms.Resize(size=(224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

# -------- Transformation for the Whole Dataset for Visualization Purposes --------This image pipeline, which processes all the data, does not work for subsequent modeling, only for the visualization below
data_transform = transforms.Compose([
    transforms.Resize(size=(224, 224)),
    transforms.ToTensor(),
])

## Loading the Data

In [ ]:
# Apply the initially categorized data to the previously defined image transformation pipeline.
train = datasets.ImageFolder(root="./splitted_data/train",
                              transform=train_transform)

val = datasets.ImageFolder(root="./splitted_data/val",
                              transform=test_val_transform)

test = datasets.ImageFolder(root="./test",
                              transform=test_val_transform)

# -------- Getting the Whole Dataset for Visualization Purposes --------
data = datasets.ImageFolder(root="./train",
                              transform=data_transform)
# print("Class to Index Mapping:", data.class_to_idx)

In [ ]:
# The order is not same, and unified processing is required here.
# for example： after
# label_dic={
#     0: "Blight",
#     1: "Common Rust",
#     2: "Gray Leaf Spot",
#     3: "Healthy",
# }
# before
# label_dic={
#     0: "Healthy",
#     1: "Common Rust",
#     2: "Blight",
#     3: "Gray Leaf Spot",
# }
new_label_dic = {v: k for k, v in label_dic.items()}

train.samples = [(path, new_label_dic[train.classes[label]]) for path, label in train.samples]
val.samples = [(path, new_label_dic[val.classes[label]]) for path, label in val.samples]
test.samples = [(path, new_label_dic[test.classes[label]]) for path, label in test.samples]

#### Plotting a few Images

Images plotted below did not undergo a data augmentation process. Therefore, they seem to be normal.

In [ ]:
labels_for_viz = {v: k for k, v in data.class_to_idx.items()}

fig, ax = plt.subplots(3, 5, figsize=(15, 10))
ax = ax.flatten()
for i in range(15):
    sample = random.randint(0, len(data))
    ax[i].imshow(data[sample][0].permute(1, 2, 0))
    ax[i].title.set_text(labels_for_viz[data[sample][1]])

### Plotting the Augmented Images

Because data augmentation methods are applied to train set, it can be seen that images are blurred, horizontally and vertically flipped compared to the unaugmented images.

In [ ]:
fig, ax = plt.subplots(3, 5, figsize=(15, 10))
ax = ax.flatten()
for i in range(15):
    sample = random.randint(0, len(train))
    ax[i].imshow(train[sample][0].permute(1, 2, 0))
    ax[i].title.set_text(labels_for_viz[train[sample][1]])

Before passing the data in a neural network, it should be batched and shuffled. Since test will not be used in training process, there is no need to shuffle it.

In [ ]:
train_dataloader = DataLoader(dataset=train,
                             batch_size=32,
                             num_workers=2,
                             shuffle=True)

val_dataloader = DataLoader(dataset=val,
                             batch_size=32,
                             num_workers=2,
                             shuffle=True)

test_dataloader = DataLoader(dataset=test,
                             batch_size=32,
                             num_workers=2,
                             shuffle=False)

When looking at the order of the data in splitted data folder, it can be observed that images from class "Blight" comes first, "Common_Rust" second, "Gray_Leaf_Spot" third and "Healthy" last. An order as such can be problematic for the model to train unbiasedly. Let's take a batch and look at the labels to make sure that the data is shuffled.

In [ ]:
img, label = next(iter(train_dataloader))
print("Batch and Image Shape:", img.shape, "--> [batch_size, color_channels, height, width]")
print("\nLabels:", label)

## Building the Deep Learning Model

### Defining Train and Test Functions

In [ ]:
def train_and_validate_model(num_epochs, model_name):
    val_best_accuracy = 0.0
    val_best_loss = float('inf')
    train_loss_history = []
    val_loss_history = []
    train_accuracy_history = []
    val_accuracy_history = []
    
    print("Training begins...")
    for epoch in range(1, num_epochs+1):
        running_train_loss = 0.0
        running_train_accuracy = 0.0
        train_total = 0
        running_val_accuracy = 0.0
        running_val_loss = 0.0
        val_total = 0
        
        
        # TRAINING LOOP
        for data in tqdm(train_dataloader):
        # training step-------
            inputs, outputs = data
            inputs, outputs = inputs.to(device), outputs.to(device)
            #Clear the previous gradients
            optimizer.zero_grad()
            #forward propagation
            predictions = model(inputs)
            # caculate loss
            train_loss = loss_fn(predictions, outputs)
            # backpropagation
            train_loss.backward()
            # upgrade parameter
            optimizer.step()
            
            # Calculate the accuracy and loss for each batch during the training process, accumulate the results, and subsequently compute the overall training accuracy and average loss for the entire epoch.
            _, train_predicted = torch.max(predictions, 1)
            running_train_accuracy += (train_predicted == outputs).sum().item()
            train_total += outputs.size(0)
            running_train_loss += train_loss.item()
            
        train_loss_value = running_train_loss/len(train_dataloader)
        train_loss_history.append(train_loss_value)
        train_accuracy = (100*running_train_accuracy)/train_total
        train_accuracy_history.append(train_accuracy)
        
        # VALIDATION LOOP
        with torch.no_grad(): #Disable gradient computation to reduce computational overhead and accelerate the validation process.
            #Set the model to evaluation mode.
            model.eval()
            for data in val_dataloader:
                inputs, outputs = data
                inputs, outputs = inputs.to(device), outputs.to(device)
                predictions = model(inputs)
                val_loss = loss_fn(predictions, outputs)
                
                _, val_predicted = torch.max(predictions, 1)
                running_val_loss += val_loss.item()
                val_total += outputs.size(0)
                running_val_accuracy += (val_predicted == outputs).sum().item()
                
        val_loss_value = running_val_loss/len(val_dataloader)
        val_loss_history.append(val_loss_value)
        val_accuracy = (100*running_val_accuracy)/val_total
        val_accuracy_history.append(val_accuracy)
        
        if val_best_loss > val_loss_value:
            torch.save(model.state_dict(), "{}_best.pth".format(model_name))
            val_best_loss = val_loss_value
        
        print("Completed Epoch: ", epoch, "- Training Accuracy: %.4f" %train_accuracy, "- Validation Accuracy: %.4f" %val_accuracy, "- Training Loss: %.4f" %train_loss_value, "- Validation Loss: %.4f" %val_loss_value)
    return train_accuracy_history, val_accuracy_history, train_loss_history, val_loss_history

In [ ]:
def test_model(model_name):
    path = "{}_best.pth".format(model_name)
    model.load_state_dict(torch.load(path))
    model.eval()
    
    running_accuracy = 0
    total = 0
    all_predictions = []
    all_true = []
    
    with torch.no_grad():
        for data in val_dataloader:
            inputs, outputs = data
            inputs, outputs = inputs.to(device), outputs.to(device)
            outputs = outputs.to(torch.float32)
            predictions = model(inputs)
            
            _, predicted = torch.max(predictions, 1)
            total += outputs.size(0)
            running_accuracy += (predicted == outputs).sum().item()
            accuracy = 100*running_accuracy/total
            
            all_predictions.append(list(predicted.to("cpu").numpy()))
            all_true.append(list(outputs.to("cpu").numpy()))
        print("Test Accuracy: ", accuracy)
    return np.hstack(all_predictions), np.hstack(all_true)

## Transfer Learning

Transfer learning is a process of utilizing pre-trained machine learning models and applying them to a related task without need of models' learning from scratch for saving time and possibly better performance.

### Resnet18

In [ ]:
model = models.resnet18(pretrained=True)
#Modify the last fully connected layer (fc) of ResNet18 to adjust the number of output classes to match the number of classes(4) in the current task.
model.fc = nn.Linear(in_features=512, out_features=4) 
model = model.to(device)

In [ ]:
# Defining the Loss Function and Choosing an Optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)  # weight_decay=1e-5  L2 regularization is not better here.

In [ ]:
EPOCHS = 30
train_accuracy_history, val_accuracy_history, train_loss_history, val_loss_history = train_and_validate_model(EPOCHS, 'resnet18')
print("Training finished...\n")
# return all predictions and true label from test model 
all_preds, all_true = test_model('resnet18')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax = ax.flatten()

ax[0].plot(train_loss_history, label="Train")
ax[0].plot(val_loss_history, label="Validation")
ax[0].title.set_text("Loss")
ax[0].legend()

ax[1].plot(train_accuracy_history, label="Train")
ax[1].plot(val_accuracy_history, label="Validation")
ax[1].title.set_text("Accuracy Score")
ax[1].legend()

plt.show()

In [ ]:
print(classification_report(all_true, all_preds, target_names=list(label_dic.values()), digits=4))

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(confusion_matrix(all_true, all_preds), cmap="Blues", annot=True, fmt="d", 
            xticklabels=list(label_dic.values()), yticklabels=list(label_dic.values()))
plt.title("Confusion Matrix", size=15)
plt.show()

### Resnet50

In [ ]:
model = models.resnet50(pretrained=True)
model.fc = nn.Linear(in_features=2048, out_features=4)
model = model.to(device)

In [ ]:
# Defining the Loss Function and Choosing an Optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4,weight_decay=1e-5)

In [ ]:
EPOCHS = 30
train_accuracy_history, val_accuracy_history, train_loss_history, val_loss_history = train_and_validate_model(EPOCHS, 'resnet50')
print("Training finished...\n")
# return all predictions and true label from test model 
all_preds, all_true = test_model('resnet50')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax = ax.flatten()

ax[0].plot(train_loss_history, label="Train")
ax[0].plot(val_loss_history, label="Validation")
ax[0].title.set_text("Loss")
ax[0].legend()

ax[1].plot(train_accuracy_history, label="Train")
ax[1].plot(val_accuracy_history, label="Validation")
ax[1].title.set_text("Accuracy Score")
ax[1].legend()

plt.show()

In [ ]:
print(classification_report(all_true, all_preds, target_names=list(label_dic.values())))

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(confusion_matrix(all_true, all_preds), cmap="Blues", annot=True, fmt="d", 
            xticklabels=list(label_dic.values()), yticklabels=list(label_dic.values()))
plt.title("Confusion Matrix", size=15)
plt.show()

### Resnet101

In [ ]:
model = models.resnet101(pretrained=True)
model.fc = nn.Linear(in_features=2048, out_features=4)
model = model.to(device)

In [ ]:
# Defining the Loss Function and Choosing an Optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4,weight_decay=1e-5)

In [ ]:
EPOCHS = 30
train_accuracy_history, val_accuracy_history, train_loss_history, val_loss_history = train_and_validate_model(EPOCHS, 'resnet101')
print("Training finished...\n")
# return all predictions and true label from test model 
all_preds, all_true = test_model('resnet101')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax = ax.flatten()

ax[0].plot(train_loss_history, label="Train")
ax[0].plot(val_loss_history, label="Validation")
ax[0].title.set_text("Loss")
ax[0].legend()

ax[1].plot(train_accuracy_history, label="Train")
ax[1].plot(val_accuracy_history, label="Validation")
ax[1].title.set_text("Accuracy Score")
ax[1].legend()

plt.show()

In [ ]:
print(classification_report(all_true, all_preds, target_names=list(label_dic.values())))

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(confusion_matrix(all_true, all_preds), cmap="Blues", annot=True, fmt="d", 
            xticklabels=list(label_dic.values()), yticklabels=list(label_dic.values()))
plt.title("Confusion Matrix", size=15)
plt.show()

### Resnet152


In [ ]:
model = models.resnet152(pretrained=True)
model.fc = nn.Linear(in_features=2048, out_features=4)
model = model.to(device)

In [ ]:
# Defining the Loss Function and Choosing an Optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4,weight_decay=1e-5)

In [ ]:
EPOCHS = 30
train_accuracy_history, val_accuracy_history, train_loss_history, val_loss_history = train_and_validate_model(EPOCHS, 'resnet152')
print("Training finished...\n")
# return all predictions and true label from test model 
all_preds, all_true = test_model('resnet152')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax = ax.flatten()

ax[0].plot(train_loss_history, label="Train")
ax[0].plot(val_loss_history, label="Validation")
ax[0].title.set_text("Loss")
ax[0].legend()

ax[1].plot(train_accuracy_history, label="Train")
ax[1].plot(val_accuracy_history, label="Validation")
ax[1].title.set_text("Accuracy Score")
ax[1].legend()

plt.show()

In [ ]:
print(classification_report(all_true, all_preds, target_names=list(label_dic.values())))

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(confusion_matrix(all_true, all_preds), cmap="Blues", annot=True, fmt="d", 
            xticklabels=list(label_dic.values()), yticklabels=list(label_dic.values()))
plt.title("Confusion Matrix", size=15)
plt.show()

### Tresnet-21K

In [ ]:
model = timm.create_model('tresnet_m.miil_in21k', pretrained=True, num_classes=4)
model = model.to(device)

In [ ]:
# Defining the Loss Function and Choosing an Optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4,weight_decay=1e-5)

In [ ]:
EPOCHS = 30
train_accuracy_history, val_accuracy_history, train_loss_history, val_loss_history = train_and_validate_model(EPOCHS, 'tresnet')
print("Training finished...\n")
# return all predictions and true label from test model 
all_preds, all_true = test_model('tresnet')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax = ax.flatten()

ax[0].plot(train_loss_history, label="Train")
ax[0].plot(val_loss_history, label="Validation")
ax[0].title.set_text("Loss")
ax[0].legend()

ax[1].plot(train_accuracy_history, label="Train")
ax[1].plot(val_accuracy_history, label="Validation")
ax[1].title.set_text("Accuracy Score")
ax[1].legend()

plt.show()

In [ ]:
print(classification_report(all_true, all_preds, target_names=list(label_dic.values())))

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(confusion_matrix(all_true, all_preds), cmap="Blues", annot=True, fmt="d", 
            xticklabels=list(label_dic.values()), yticklabels=list(label_dic.values()))
plt.title("Confusion Matrix", size=15)
plt.show()

### ViT-B-16-21K

In [ ]:
model = timm.create_model('vit_base_patch16_224_miil.in21k', pretrained=True, num_classes=4)
model = model.to(device)

In [ ]:
# Defining the Loss Function and Choosing an Optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4,weight_decay=1e-5)

In [ ]:
EPOCHS = 30
train_accuracy_history, val_accuracy_history, train_loss_history, val_loss_history = train_and_validate_model(EPOCHS, 'vit_base')
print("Training finished...\n")
# return all predictions and true label from test model 
all_preds, all_true = test_model('vit_base')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax = ax.flatten()

ax[0].plot(train_loss_history, label="Train")
ax[0].plot(val_loss_history, label="Validation")
ax[0].title.set_text("Loss")
ax[0].legend()

ax[1].plot(train_accuracy_history, label="Train")
ax[1].plot(val_accuracy_history, label="Validation")
ax[1].title.set_text("Accuracy Score")
ax[1].legend()

plt.show()

In [ ]:
print(classification_report(all_true, all_preds, target_names=list(label_dic.values())))

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(confusion_matrix(all_true, all_preds), cmap="Blues", annot=True, fmt="d", 
            xticklabels=list(label_dic.values()), yticklabels=list(label_dic.values()))
plt.title("Confusion Matrix", size=15)
plt.show()

### Resnext101

In [ ]:
model = models.resnext101_64x4d(pretrained=True)
model.fc = nn.Linear(in_features=2048, out_features=4)
model = model.to(device)

In [ ]:
# Defining the Loss Function and Choosing an Optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4,weight_decay=1e-5)

In [ ]:
EPOCHS = 30
train_accuracy_history, val_accuracy_history, train_loss_history, val_loss_history = train_and_validate_model(EPOCHS, 'resnext101')
print("Training finished...\n")
# return all predictions and true label from test model 
all_preds, all_true = test_model('resnext101')
# Here, I was running the model with added regularization (after the leaderboard deadline), but the training speed was too slow, and I needed to submit the assignment, so I had to pause it. However, I didn’t choose this model previously, and my last submission file shows that its performance wasn’t best.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax = ax.flatten()

ax[0].plot(train_loss_history, label="Train")
ax[0].plot(val_loss_history, label="Validation")
ax[0].title.set_text("Loss")
ax[0].legend()

ax[1].plot(train_accuracy_history, label="Train")
ax[1].plot(val_accuracy_history, label="Validation")
ax[1].title.set_text("Accuracy Score")
ax[1].legend()

plt.show()

In [ ]:
print(classification_report(all_true, all_preds, target_names=list(label_dic.values())))

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(confusion_matrix(all_true, all_preds), cmap="Blues", annot=True, fmt="d", 
            xticklabels=list(label_dic.values()), yticklabels=list(label_dic.values()))
plt.title("Confusion Matrix", size=15)
plt.show()

## Submission

In [ ]:
# choose the best model
model = timm.create_model('vit_base_patch16_224_miil.in21k', pretrained=True, num_classes=4)
model = model.to(device)
path = "./vit_base_best.pth"
model.load_state_dict(torch.load(path))
model.eval()

all_predictions = []

with torch.no_grad():
    for data in test_dataloader:
        inputs, outputs = data
        inputs, outputs = inputs.to(device), outputs.to(device)
        outputs = outputs.to(torch.float32)
        predictions = model(inputs)
        _, predicted = torch.max(predictions, 1)
        
        all_predictions.append(list(predicted.to("cpu").numpy()))
test_preds = np.hstack(all_predictions)

In [ ]:
submission = pd.DataFrame([[int(_[0].split('\\')[-1].split('.')[0]) for _ in test.samples], test_preds]).T.sort_values(0)
submission.columns = ["Image", "Predicted"]
submission.to_csv('submission.csv', index=False)